# Train, evaluate and export a run

Run on a GPU runtime. Set the parameters in the next cell, then run the cells in order: setup, stage the shards and eval plots, throughput check, release baseline (once), train, evaluate, export, run comparison, close.

Re-running the train cell after a disconnect resumes from the last checkpoint on Drive.

The repo is private: add a read-only GitHub token (Contents: read) under Colab Secrets (the key icon) as `GITHUB_TOKEN`, with notebook access on.

In [ ]:
RUN_ID = "neon-platform"
BASELINE_ID = "release-baseline"  # the released weights, scored once on the same benchmark
CONFIG = "experiments/configs/neon_platform.yaml"
REPO_SHA = ""
SOURCES = ["neon"]
WORKERS = 4  # loader workers: the fastest count in throughput.json (4 on an L4)

In [ ]:
import subprocess
import sys
from pathlib import Path

from google.colab import drive, userdata

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/fair_trees")

assert REPO_SHA, "set REPO_SHA to the pushed commit"
# the repo is private: a read-only GitHub token in Colab Secrets (key icon), named GITHUB_TOKEN
REPO = "github.com/OmdenaAI/HOTOSM-fAIr-Trees.git"
token = userdata.get("GITHUB_TOKEN")
!rm -rf /content/fair_trees
# git clone through subprocess so the token never reaches the cell output
url = f"https://x-access-token:{token}@{REPO}"
clone = subprocess.run(["git", "clone", "-q", url, "/content/fair_trees"])
# checked by hand: a CalledProcessError message would print the URL, token included
if clone.returncode:
    raise RuntimeError(f"git clone failed (exit {clone.returncode}): check the GITHUB_TOKEN secret")
# drop the token from the saved remote so it is not left in .git/config
git = ["git", "-C", "/content/fair_trees"]
subprocess.run([*git, "remote", "set-url", "origin", f"https://{REPO}"], check=True)
subprocess.run([*git, "checkout", "-q", REPO_SHA], check=True)
del token, url

%pip install "/content/fair_trees[train,test]" -r /content/fair_trees/experiments/requirements.txt
%cd /content/fair_trees
sys.path.insert(0, "/content/fair_trees")

In [ ]:
import json

import torch
import torchvision

from experiments import run

assert torch.__version__.split("+")[0] == "2.11.0" and torchvision.__version__.split("+")[0] == "0.26.0"
print(torch.cuda.get_device_name())

In [ ]:
import zipfile

from experiments.shards import stage_shards

stage_shards(DRIVE, Path("/content/data"), SOURCES)

# the eval plots come from the annotation and evaluation archives kept on Drive; only the labels
# and the RGB plots are extracted (evaluation.zip also holds the LiDAR)
parts = ("annotations", "evaluation/RGB")
raw_drive = DRIVE / "raw" / "neon"
listed = {f["name"] for f in json.loads((raw_drive / "manifest.json").read_text())["files"]}
eval_root = Path("/content/data/NeonTreeEvaluation")
eval_root.mkdir(parents=True, exist_ok=True)
for name in ("annotations.zip", "evaluation.zip"):
    assert name in listed, f"{name} is not in the raw manifest on Drive"
    with zipfile.ZipFile(raw_drive / name) as z:
        members = [m for m in z.namelist() if m.startswith(tuple(f"{p}/" for p in parts))]
        z.extractall(eval_root, members=members)
for part in parts:
    assert (eval_root / part).is_dir(), f"{eval_root / part} is missing after extraction"

In [ ]:
!python -m experiments.throughput $CONFIG $RUN_ID
throughput = json.loads((DRIVE / "runs" / RUN_ID / "throughput.json").read_text())
print(json.dumps(throughput, indent=2))
print("fastest loader workers:", max(throughput["loader"], key=throughput["loader"].get))
assert throughput["pass"], "loader is slower than the GPU step: stop here and report throughput.json"

Release baseline: run once. It scores the released weights on the same benchmark.

In [ ]:
run.evaluate(
    BASELINE_ID,
    Path("experiments/benchmark.json"),
    config=Path(CONFIG),
    weights=DRIVE / "weights" / "NEON.pt",
)

In [ ]:
result = run.main(Path(CONFIG), RUN_ID, num_workers=WORKERS)  # re-running this cell resumes
print(result.best_metrics)

In [ ]:
report = run.evaluate(RUN_ID, Path("experiments/benchmark.json"))
neon = report["parts"]["neon"]
print(neon["f1"], neon["f1_ci95"], neon["forgetting_pass"])

In [ ]:
exported = run.export(RUN_ID, Path("experiments/benchmark.json"))
print(exported)
assert exported["pass"]

The resume check compares two runs of `experiments/configs/neon_resume_check.yaml`. To produce them:

1. In the parameters cell set `RUN_ID = "neon-resume-a"` and `CONFIG` to that file, run the setup, torch check, stage and train cells uninterrupted, then evaluate it.
2. Set `RUN_ID = "neon-resume-b"` and run the train cell. During epoch 1, once a mid-epoch checkpoint `ckpt-e001-s<step>.pt` with a nonzero step (not `ckpt-e001-s000000.pt`) appears in `runs/neon-resume-b` on Drive, disconnect and delete the runtime.
3. Reconnect and rerun, in order: the parameters cell (with `RUN_ID = "neon-resume-b"` and the same `CONFIG`), setup, torch check, stage, then the same train cell. Evaluate `neon-resume-b`.
4. Compare the two runs below.

In [ ]:
print(run.compare_runs("neon-resume-a", "neon-resume-b"))
print(run.compare_runs(BASELINE_ID, RUN_ID))

Optional: weight interpolation. Blends the base weights with this run's `best.pt` (`(1 - alpha) * base + alpha * tuned`) and scores each blend on this run's val regions and the NEON benchmark, to see whether a point between them keeps the val gain without the forgetting. Run after the evaluate cell; each blend is its own run folder `<RUN_ID>-wise-<alpha>`.

In [ ]:
from experiments.interpolate import interpolate

ALPHAS = (0.3, 0.5, 0.55, 0.6, 0.7)  # share of the fine-tuned weights
base_weights = DRIVE / run.load_config(Path(CONFIG))["weights"]
for alpha in ALPHAS:
    blend_id = f"{RUN_ID}-wise-{alpha:g}"
    blend = interpolate(
        base_weights, DRIVE / "runs" / RUN_ID / "best.pt", alpha, DRIVE / "runs" / blend_id / "weights.pt"
    )
    benchmark = Path("experiments/benchmark.json")
    blend_report = run.evaluate(blend_id, benchmark, config=Path(CONFIG), weights=blend)
    val, neon = blend_report["parts"]["val"], blend_report["parts"]["neon"]
    print(f"{blend_id}: val F1 {val['f1']:.3f}, NEON F1 {neon['f1']:.3f} (pass {neon['forgetting_pass']})")
    print(run.compare_runs(RUN_ID, blend_id)["paired_diff"])

In [ ]:
drive.flush_and_unmount()